# 05 · Case study: a real-like signal / 案例研究

This notebook walks through a realistic scenario: a low-SNR burst riding on coloured background, using CEEMDAN to avoid mode mixing.

**Reproducibility note**: the raw 'real' file under `examples/sample_signal.csv` is a synthetic mixture (no licensing concerns). Swap the `np.loadtxt` call with your own data to apply the workflow.

**可重現性**：`examples/sample_signal.csv` 是合成資料（避免授權問題）。把 `np.loadtxt` 換成你自己的資料即可套用此流程。

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt
from PyEMD import CEEMDAN

from emdsig import (
    compute_et, chi2_confidence_bounds, monte_carlo_bounds,
    calibrate_intercept, plot_significance,
)

## 1. Load the sample signal / 讀取範例資料

In [ ]:
x = np.loadtxt(pathlib.Path('..') / 'examples' / 'sample_signal.csv')
N = len(x)
print(f'length = {N}')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(x, lw=0.6); ax.set_title('Input time series')
plt.show()

## 2. Decompose with CEEMDAN / 用 CEEMDAN 分解

CEEMDAN suppresses mode mixing, which is critical for reliable (E, T) estimates.

In [ ]:
ceemdan = CEEMDAN(trials=20, epsilon=0.05)   # trials small for speed
imfs = ceemdan.ceemdan(x)
print(f'IMFs: {imfs.shape[0]}')

## 3. Significance test (closed-form) / 閉式檢定

In [ ]:
E, T = compute_et(imfs, trim_ratio=0.05)
valid = ~np.isnan(T)
lnE, lnT = np.log(E[valid]), np.log(T[valid])

c = calibrate_intercept(lnT[0], lnE[0])
grid = np.linspace(lnT.min() - 0.5, lnT.max() + 0.5, 200)
lo, hi = chi2_confidence_bounds(grid, N=N, alpha=0.05, baseline_intercept=c)

plot_significance(lnT, lnE, bounds=(grid, lo, hi), baseline_intercept=c,
                  title='Case study — chi-squared 95% CI')
plt.show()

## 4. Reconstruct the signal using only significant IMFs / 只用顯著 IMF 重建訊號

In [ ]:
hi_at_points = np.interp(lnT, grid, hi)
mask = lnE > hi_at_points            # significant IMFs

valid_idx = np.where(valid)[0]
sig_imf_indices = valid_idx[mask]
print(f'significant IMFs: {sig_imf_indices.tolist()}')

reconstruction = imfs[sig_imf_indices].sum(axis=0)

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(x, lw=0.5, alpha=0.5, label='original')
ax.plot(reconstruction, lw=1.0, label='significant-IMF reconstruction')
ax.legend(); ax.set_title('Denoised reconstruction')
plt.show()

## 5. Optional: cross-check with Monte Carlo / 可選：Monte Carlo 交叉驗證

Use this when you are not sure the background is white — see notebook 04.

In [ ]:
# Uncomment the next block to run (slow; ~tens of seconds)
# grid_mc, lo_mc, hi_mc = monte_carlo_bounds(N=N, n_trials=50, noise='white', seed=42)
# ax = plot_significance(lnT, lnE, bounds=(grid_mc, lo_mc, hi_mc),
#                        title='Monte Carlo cross-check')
# plt.show()